In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the cleaned dataset from Week 1
# Note: Update the path below if your CSV is stored inside a specific folder (e.g., 'Data/cleaned_travel_data.csv')
df = pd.read_csv('../data/cleaned_travel_data.csv')

# 2. Separate the Target (what we want to predict) from the Features (the inputs)
X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

# 3. Verify the shapes to ensure it loaded correctly
print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

Features (X) shape: (87396, 32)
Target (y) shape: (87396,)


In [3]:
# 1. One-Hot Encode Categorical Variables
# The 'drop_first=True' argument prevents the dummy variable trap (multicollinearity)
X_encoded = pd.get_dummies(X, drop_first=True)

# 2. Split the Data into Training (80%) and Testing (20%) sets
# random_state=42 ensures your results are reproducible every time you run this notebook
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 3. Verify the new shapes after encoding and splitting
print(f"Training Features shape: {X_train.shape}")
print(f"Testing Features shape: {X_test.shape}")

Training Features shape: (69916, 1972)
Testing Features shape: (17480, 1972)


In [4]:
# Identify categorical columns (text data)
categorical_cols = X.select_dtypes(include=['object']).columns

# Print the number of unique values in each categorical column
print("Unique values per categorical column:")
print(X[categorical_cols].nunique())

Unique values per categorical column:
hotel                        2
arrival_date_month          12
meal                         5
country                    178
market_segment               8
distribution_channel         5
reserved_room_type          10
assigned_room_type          12
deposit_type                 3
customer_type                4
reservation_status           3
reservation_status_date    926
arrival_date_full          793
arrival_day_of_week          7
dtype: int64


C:\Users\payal\AppData\Local\Temp\ipykernel_25468\193686615.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns


In [5]:
# 1. Drop the columns causing Data Leakage and High Cardinality
cols_to_drop = ['reservation_status', 'reservation_status_date', 'arrival_date_full', 'country']
X_clean = X.drop(columns=cols_to_drop)

# 2. Re-run One-Hot Encoding on the clean dataset
X_encoded = pd.get_dummies(X_clean, drop_first=True)

# 3. Split the Data into Training (80%) and Testing (20%) sets again
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 4. Verify the new shapes
print(f"Training Features shape: {X_train.shape}")
print(f"Testing Features shape: {X_test.shape}")

Training Features shape: (69916, 76)
Testing Features shape: (17480, 76)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Initialize the model
# Setting max_iter=1000 gives the algorithm enough time to converge on this size dataset
model = LogisticRegression(max_iter=1000, random_state=42)

# 2. Train the model using the training data
print("Training the model... (this may take a few seconds)")
model.fit(X_train, y_train)

# 3. Make predictions on the unseen testing data
y_pred = model.predict(X_test)

# 4. Evaluate the model's performance
print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training the model... (this may take a few seconds)

--- Model Evaluation ---
Accuracy: 0.7859

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.92      0.86     12733
           1       0.66      0.44      0.53      4747

    accuracy                           0.79     17480
   macro avg       0.74      0.68      0.69     17480
weighted avg       0.77      0.79      0.77     17480



c:\Users\payal\Project-2-Travel-Tourism-Hospitality---Customer-Retention-and-Dynamic-Pricing-Analysis\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
from sklearn.preprocessing import StandardScaler

# 1. Initialize the Scaler
scaler = StandardScaler()

# 2. Scale the data
# We 'fit' (learn the math) ONLY on the training data to prevent data leakage, 
# then 'transform' (apply the math) to both train and test sets.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Retrain the model on the newly scaled data
print("Retraining the model on scaled data...")
model_scaled = LogisticRegression(max_iter=1000, random_state=42)
model_scaled.fit(X_train_scaled, y_train)

# 4. Make new predictions and evaluate
y_pred_scaled = model_scaled.predict(X_test_scaled)

print("\n--- New Model Evaluation (Scaled) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_scaled):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_scaled))

Retraining the model on scaled data...

--- New Model Evaluation (Scaled) ---
Accuracy: 0.7942

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.93      0.87     12733
           1       0.70      0.43      0.53      4747

    accuracy                           0.79     17480
   macro avg       0.76      0.68      0.70     17480
weighted avg       0.78      0.79      0.78     17480



In [8]:
# 1. Initialize the model with balanced class weights
print("Retraining the model with balanced class weights...")
model_balanced = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# 2. Train the model on the scaled training data
model_balanced.fit(X_train_scaled, y_train)

# 3. Make predictions on the scaled testing data
y_pred_balanced = model_balanced.predict(X_test_scaled)

# 4. Evaluate the new balanced model
print("\n--- New Model Evaluation (Balanced & Scaled) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_balanced):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced))

Retraining the model with balanced class weights...

--- New Model Evaluation (Balanced & Scaled) ---
Accuracy: 0.7053

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.68      0.77     12733
           1       0.47      0.77      0.59      4747

    accuracy                           0.71     17480
   macro avg       0.68      0.73      0.68     17480
weighted avg       0.78      0.71      0.72     17480



In [9]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Extract the coefficients (weights) from the model
feature_names = X_train.columns
coefficients = model_balanced.coef_[0]

# 2. Create a DataFrame to organize the data
feature_importance = pd.DataFrame({
    'Feature': feature_names, 
    'Importance': coefficients
})

# 3. Sort by the most impactful features
# Positive importance means it DRIVES cancellations. Negative means it PREVENTS them.
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

# 4. Display the Top 10 Drivers OF Cancellation
print("--- Top 10 Features Driving Cancellations ---")
print(feature_importance.head(10).to_string(index=False))

# 5. Display the Top 10 Features PREVENTING Cancellations
print("\n--- Top 10 Features Preventing Cancellations (Safe Bookings) ---")
print(feature_importance.tail(10).to_string(index=False))

--- Top 10 Features Driving Cancellations ---
                 Feature  Importance
  previous_cancellations    0.769788
    reserved_room_type_E    0.496040
    reserved_room_type_D    0.465316
    reserved_room_type_G    0.421417
 deposit_type_Non Refund    0.398286
               lead_time    0.397868
    reserved_room_type_F    0.348161
                     adr    0.317226
 customer_type_Transient    0.260055
market_segment_Online TA    0.158988

--- Top 10 Features Preventing Cancellations (Safe Bookings) ---
                       Feature  Importance
          assigned_room_type_C   -0.203970
          assigned_room_type_I   -0.271548
previous_bookings_not_canceled   -0.391799
  market_segment_Offline TA/TO   -0.401324
          assigned_room_type_F   -0.502810
     total_of_special_requests   -0.522422
          assigned_room_type_G   -0.547495
          assigned_room_type_D   -0.574198
          assigned_room_type_E   -0.574663
   required_car_parking_spaces   -5.636150
